# Level Reversal EA: Backtest & Optimization

Бэктест и оптимизация стратегии отскока от уровней с подтверждением разворотным паттерном.

**Требование к данным:** CSV файлы в `content/` для EURUSD по таймфреймам M5, M15, H4, D1 с колонками `Open, High, Low, Close, Volume` и индексом времени (datetime).

In [19]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print('Импорт базовых библиотек успешен')

Импорт базовых библиотек успешен


In [20]:
# Установка backtesting.py если нужно
import subprocess
import sys

try:
    from backtesting import Backtest, Strategy
    from backtesting.lib import crossover
    from backtesting.test import SMA, GOOG
except ImportError:
    print('Установка backtesting.py...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'backtesting'])
    from backtesting import Backtest, Strategy
    from backtesting.lib import crossover
    from backtesting.test import SMA, GOOG

print('backtesting.py готов к использованию')

backtesting.py готов к использованию


## 1. Загрузка и подготовка данных

In [21]:
import os
from pathlib import Path

data_dir = Path('/workspaces/FV_Bounce/content')
print('Доступные файлы в content/:')
for f in data_dir.glob('*.csv'):
    print(f'  - {f.name} ({f.stat().st_size / 1024:.1f} KB)')

Доступные файлы в content/:
  - EURUSD_M5_2024-01-01_2025-12-31.csv (3059.0 KB)
  - EURUSD_M15_2024-01-01_2025-12-31.csv (2833.9 KB)
  - EURUSD_H4_2024-01-01_2025-12-31.csv (181.3 KB)
  - EURUSD_D1_2024-01-01_2025-12-31.csv (30.6 KB)


In [22]:
def load_csv_data(filepath):
    """
    Загрузка CSV данных, преобразование индекса в datetime.
    Ожидаемые колонки: Open, High, Low, Close, Volume (или Open, Close, High, Low, Volume).
    """
    df = pd.read_csv(filepath)
    
    # Определение колонки времени (может быть 'Time', 'Datetime', '<TIME>' или первая колонка)
    time_cols = [col for col in df.columns if col.lower() in ['time', 'datetime', '<time>']]
    if time_cols:
        time_col = time_cols[0]
    else:
        time_col = df.columns[0]  # предполагаем первая колонка - время
    
    df[time_col] = pd.to_datetime(df[time_col])
    df = df.set_index(time_col)
    df = df.sort_index()
    
    # Стандартизация имён колонок
    df.columns = [col.lower().strip() for col in df.columns]
    
    # Переименование если нужно
    rename_map = {
        'o': 'open', 'h': 'high', 'l': 'low', 'c': 'close', 'v': 'volume',
        '<open>': 'open', '<high>': 'high', '<low>': 'low', '<close>': 'close', '<tick_volume>': 'volume'
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)
    
    # Удаление NaN
    df = df.dropna(subset=['open', 'high', 'low', 'close'])
    
    # Возвращаем с заглавными буквами (требуется для backtesting.py)
    return df[['open', 'high', 'low', 'close', 'volume']].rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume'
    })

# Попытка загрузить M5 данные
m5_files = list(data_dir.glob('*M5*.csv')) + list(data_dir.glob('*m5*.csv'))
if m5_files:
    print(f'Найден M5 файл: {m5_files[0].name}')
    df_m5 = load_csv_data(str(m5_files[0]))
    print(f'M5: {len(df_m5)} баров, период {df_m5.index[0]} - {df_m5.index[-1]}')
    print(df_m5.head())
else:
    print('⚠️  M5 данные не найдены. Пожалуйста, загрузите файл типа EURUSD_M5_*.csv в content/')

Найден M5 файл: EURUSD_M5_2024-01-01_2025-12-31.csv
M5: 53020 баров, период 2025-04-15 21:05:00 - 2025-12-31 23:55:00
                        Open     High      Low    Close  Volume
Time                                                           
2025-04-15 21:05:00  1.12694  1.12704  1.12680  1.12694     223
2025-04-15 21:10:00  1.12694  1.12707  1.12678  1.12683     225
2025-04-15 21:15:00  1.12684  1.12715  1.12675  1.12704     261
2025-04-15 21:20:00  1.12703  1.12739  1.12695  1.12698     219
2025-04-15 21:25:00  1.12694  1.12762  1.12689  1.12746     260


In [23]:
# Загрузка M15
m15_files = list(data_dir.glob('*M15*.csv')) + list(data_dir.glob('*m15*.csv'))
if m15_files:
    print(f'Найден M15 файл: {m15_files[0].name}')
    df_m15 = load_csv_data(str(m15_files[0]))
    print(f'M15: {len(df_m15)} баров, период {df_m15.index[0]} - {df_m15.index[-1]}')
else:
    print('⚠️  M15 данные не найдены. Пожалуйста, загрузите файл типа EURUSD_M15_*.csv в content/')

Найден M15 файл: EURUSD_M15_2024-01-01_2025-12-31.csv
M15: 49654 баров, период 2024-01-02 00:00:00 - 2025-12-31 23:45:00


In [24]:
# Загрузка H4 (или ресемпл из M5)
h4_files = list(data_dir.glob('*H4*.csv')) + list(data_dir.glob('*h4*.csv'))
if h4_files:
    print(f'Найден H4 файл: {h4_files[0].name}')
    df_h4 = load_csv_data(str(h4_files[0]))
    print(f'H4: {len(df_h4)} баров')
elif 'm5_files':
    print('H4 файл не найден, создаю ресемпл из M5...')
    df_h4 = df_m5.resample('4H').agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum'
    }).dropna()
    print(f'H4: {len(df_h4)} баров (ресемпл из M5)')
else:
    print('⚠️  H4 данные невозможно создать. Загрузите M5 или H4 CSV.')

Найден H4 файл: EURUSD_H4_2024-01-01_2025-12-31.csv
H4: 3111 баров


In [25]:
# Загрузка D1 (или ресемпл из M5)
d1_files = list(data_dir.glob('*D1*.csv')) + list(data_dir.glob('*d1*.csv')) + list(data_dir.glob('*_D_*.csv'))
if d1_files:
    print(f'Найден D1 файл: {d1_files[0].name}')
    df_d1 = load_csv_data(str(d1_files[0]))
    print(f'D1: {len(df_d1)} баров')
elif 'm5_files':
    print('D1 файл не найден, создаю ресемпл из M5...')
    df_d1 = df_m5.resample('1D').agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum'
    }).dropna()
    print(f'D1: {len(df_d1)} баров (ресемпл из M5)')
else:
    print('⚠️  D1 данные невозможно создать. Загрузите M5 или D1 CSV.')

Найден D1 файл: EURUSD_D1_2024-01-01_2025-12-31.csv
D1: 520 баров


## 2. Реализация компонентов стратегии

In [26]:
def calculate_atr(df, period=14):
    """Вычисление Average True Range."""
    high = df['High']
    low = df['Low']
    close = df['Close']
    
    tr1 = high - low
    tr2 = abs(high - close.shift())
    tr3 = abs(low - close.shift())
    
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(period).mean()
    return atr

def find_swing_levels(df, depth=2, lookback=250):
    """
    Поиск swing high/low (локальные экстремумы = фракталы).
    Возвращает DataFrame с колонками 'High_level' и 'Low_level'.
    """
    df = df.iloc[-lookback:].copy()
    
    high = df['High'].values
    low = df['Low'].values
    
    swing_high = np.full_like(high, np.nan)
    swing_low = np.full_like(low, np.nan)
    
    for i in range(depth, len(high) - depth):
        if all(high[i] >= high[i-j] for j in range(1, depth+1)) and \
           all(high[i] >= high[i+j] for j in range(1, depth+1)):
            swing_high[i] = high[i]
        
        if all(low[i] <= low[i-j] for j in range(1, depth+1)) and \
           all(low[i] <= low[i+j] for j in range(1, depth+1)):
            swing_low[i] = low[i]
    
    result = pd.DataFrame({
        'High_level': swing_high,
        'Low_level': swing_low
    }, index=df.index)
    
    return result

def calculate_trend_filter(df, ema_fast=50, ema_slow=200, adx_period=14, adx_threshold=20):
    """
    Определение тренда/флэта.
    Возвращает Series: 1 = восходящий, -1 = нисходящий, 0 = флэт.
    """
    close = df['Close'].values
    high = df['High'].values
    low = df['Low'].values
    
    # EMA
    ema_f = pd.Series(close).ewm(span=ema_fast).mean().values
    ema_s = pd.Series(close).ewm(span=ema_slow).mean().values
    
    # ADX (упрощённо через DX)
    plus_dm = np.maximum(high[1:] - high[:-1], 0)
    minus_dm = np.maximum(low[:-1] - low[1:], 0)
    
    tr1 = high[1:] - low[1:]
    tr2 = np.abs(high[1:] - close[:-1])
    tr3 = np.abs(low[1:] - close[:-1])
    tr = np.maximum(tr1, np.maximum(tr2, tr3))
    
    di_plus = 100 * pd.Series(plus_dm).rolling(adx_period).sum() / pd.Series(tr).rolling(adx_period).sum()
    di_minus = 100 * pd.Series(minus_dm).rolling(adx_period).sum() / pd.Series(tr).rolling(adx_period).sum()
    dx = 100 * np.abs(di_plus - di_minus) / (di_plus + di_minus)
    adx = dx.rolling(adx_period).mean()
    
    # Определение тренда
    trend = np.zeros(len(close))
    
    for i in range(max(ema_fast, len(adx))):
        if pd.isna(adx.iloc[i]) or adx.iloc[i] < adx_threshold:
            trend[i] = 0  # флэт
        elif ema_f[i] > ema_s[i]:
            trend[i] = 1   # восходящий
        else:
            trend[i] = -1  # нисходящий
    
    return pd.Series(trend, index=df.index)

def detect_pin_bar(open_p, high, low, close):
    """Детектирование pin bar паттерна."""
    body = abs(close - open_p)
    total_range = high - low
    
    if total_range == 0:
        return False
    
    body_ratio = body / total_range
    
    if body_ratio > 0.3:
        return False
    
    upper_shadow = high - max(open_p, close)
    lower_shadow = min(open_p, close) - low
    
    return (upper_shadow > 2 * lower_shadow) or (lower_shadow > 2 * upper_shadow)

def detect_engulfing(prev_open, prev_close, curr_open, curr_close, curr_high, curr_low):
    """Детектирование engulfing паттерна."""
    prev_body_high = max(prev_open, prev_close)
    prev_body_low = min(prev_open, prev_close)
    curr_body_high = max(curr_open, curr_close)
    curr_body_low = min(curr_open, curr_close)
    
    return (curr_body_high > prev_body_high) and (curr_body_low < prev_body_low)

def detect_doji(open_p, high, low, close):
    """Детектирование doji паттерна."""
    body = abs(close - open_p)
    total_range = high - low
    
    if total_range == 0:
        return False
    
    body_ratio = body / total_range
    return body_ratio < 0.1

## 3. Определение разворотных паттернов

## 4. Класс стратегии для backtesting.py

In [27]:
from backtesting import Backtest, Strategy

class LevelReversalStrategy(Strategy):
    # Параметры оптимизации
    atr_period = 14
    min_touches = 3
    level_buffer_atr = 1.0
    entry_zone_atr = 0.5
    fractal_depth = 2
    
    ema_fast = 50
    ema_slow = 200
    adx_threshold = 20
    
    sl_atr_mult = 2.0
    rr = 2.0  # Risk/Reward ratio: TP = SL * RR
    
    use_pin_bar = True
    use_engulfing = True
    use_doji = True
    
    def init(self):
        # Расчёт ATR
        self.atr = self.I(calculate_atr, self.data.df, self.atr_period)
        
        # Расчёт тренда
        self.trend = self.I(calculate_trend_filter, 
                            self.data.df, 
                            self.ema_fast, 
                            self.ema_slow, 
                            adx_threshold=self.adx_threshold)
        
        self.last_signal_bar = -1
    
    def next(self):
        if len(self.data) < self.atr_period + 5:
            return
        
        # Уже есть открытая позиция
        if self.position:
            return
        
        atr_val = self.atr[-1]
        if atr_val <= 0:
            return
        
        trend = self.trend[-1]
        close = self.data.Close[-1]
        
        # Получение уровней из предыдущего бара H4/D1 (упрощённо используем последние свинги)
        swings = find_swing_levels(self.data.df.iloc[-100:], self.fractal_depth)
        
        # Ищем ближайшую поддержку для лонга
        support_levels = swings['Low_level'].dropna().values
        if len(support_levels) > 0:
            nearest_support = support_levels[-1]
            entry_zone_dist = self.entry_zone_atr * atr_val
            
            # Проверка, находится ли цена в зоне уровня
            if abs(close - nearest_support) <= entry_zone_dist and \
               (trend == 1 or trend == 0):  # восходящий тренд или флэт
                
                # Проверка паттерна на последней свече
                o, h, l, c = self.data.Open[-1], self.data.High[-1], self.data.Low[-1], self.data.Close[-1]
                prev_c = self.data.Close[-2] if len(self.data) > 1 else c
                prev_o = self.data.Open[-2] if len(self.data) > 1 else o
                
                pin_bar = detect_pin_bar(o, h, l, c) if self.use_pin_bar else 0
                engulfing = detect_engulfing(prev_o, prev_c, o, c, h, l) if self.use_engulfing else 0
                doji = detect_doji(o, h, l, c) if self.use_doji else 0
                
                pattern_signal = max(pin_bar, engulfing, doji)
                
                if pattern_signal > 0:  # Обнаружен паттерн
                    sl_distance = self.sl_atr_mult * atr_val
                    sl_price = close - sl_distance
                    tp_price = close + sl_distance * self.rr
                    
                    self.buy(sl=sl_price, tp=tp_price)
                    self.last_signal_bar = len(self.data)
        
        # Ищем ближайшее сопротивление для шорта
        resistance_levels = swings['High_level'].dropna().values
        if len(resistance_levels) > 0:
            nearest_resistance = resistance_levels[-1]
            entry_zone_dist = self.entry_zone_atr * atr_val
            
            if abs(close - nearest_resistance) <= entry_zone_dist and \
               (trend == -1 or trend == 0):  # нисходящий тренд или флэт
                
                o, h, l, c = self.data.Open[-1], self.data.High[-1], self.data.Low[-1], self.data.Close[-1]
                prev_c = self.data.Close[-2] if len(self.data) > 1 else c
                prev_o = self.data.Open[-2] if len(self.data) > 1 else o
                
                pin_bar = detect_pin_bar(o, h, l, c) if self.use_pin_bar else 0
                engulfing = detect_engulfing(prev_o, prev_c, o, c, h, l) if self.use_engulfing else 0
                doji = detect_doji(o, h, l, c) if self.use_doji else 0
                
                pattern_signal = min(pin_bar, engulfing, doji)
                
                if pattern_signal < 0:  # Обнаружен паттерн
                    sl_distance = self.sl_atr_mult * atr_val
                    sl_price = close + sl_distance
                    tp_price = close - sl_distance * self.rr
                    
                    self.sell(sl=sl_price, tp=tp_price)
                    self.last_signal_bar = len(self.data)

print('Класс LevelReversalStrategy определён')

Класс LevelReversalStrategy определён


## 5. Бэктест и оптимизация

In [28]:
# Используем M5 данные для входа (можно переключиться на M15)
if 'df_m5' in locals() and len(df_m5) > 100:
    bt = Backtest(df_m5, LevelReversalStrategy, 
                   cash=700,      # начальный депозит
                   commission=0.0002,  # комиссия 0.02% на сторону
                   margin=0.02,   # маржа 2%
                   trade_on_close=False)  # вход на открытии следующей свечи
    
    print('Запуск начального бэктеста на M5 с параметрами по умолчанию...')
    stats = bt.run()
    print(stats)
else:
    print('⚠️  M5 данные не загружены. Пожалуйста, загрузите данные сначала.')

Запуск начального бэктеста на M5 с параметрами по умолчанию...


Start                     2025-04-15 21:05:00
End                       2025-12-31 23:55:00
Duration                    260 days 02:50:00
Exposure Time [%]                    26.38627
Equity Final [$]                      0.02294
Equity Peak [$]                     856.79605
Commissions [$]                     830.40703
Return [%]                          -99.99672
Buy & Hold Return [%]                 4.12493
Return (Ann.) [%]                   -99.99992
Volatility (Ann.) [%]                 0.00037
CAGR [%]                            -99.99995
Sharpe Ratio                    -270408.51789
Sortino Ratio                        -0.63463
Calmar Ratio                         -1.00003
Alpha [%]                          -145.94257
Beta                                 11.13858
Max. Drawdown [%]                   -99.99732
Avg. Drawdown [%]                   -14.48529
Max. Drawdown Duration      259 days 13:30:00
Avg. Drawdown Duration       32 days 11:10:00
# Trades                          

In [29]:
# Оптимизация параметров (раскомментировать для полного запуска, требует времени)
# ВАЖНО: это займёт длительное время с большим пространством параметров

if 'bt' in locals():
    print('Подготовка к оптимизации параметров...')
    print('Будут оптимизированы: sl_atr_mult, rr, min_touches, level_buffer_atr')
    
    # Раскомментировать для запуска
    # opt_stats = bt.optimize(
    #     sl_atr_mult=[1.5, 2.0, 2.5, 3.0],
    #     rr=[1.5, 2.0, 2.5, 3.0],
    #     min_touches=[2, 3, 4],
    #     level_buffer_atr=[0.5, 1.0, 1.5],
    #     maximize='Sharpe Ratio',  # или 'Return [%]', 'Max. Drawdown [%]'
    #     constraint=lambda p: p.sl_atr_mult * p.rr <= 10,  # SL * RR не более 10 ATR
    #     processes=4
    # )
    # print('Оптимизация завершена:')
    # print(opt_stats)

Подготовка к оптимизации параметров...
Будут оптимизированы: sl_atr_mult, rr, min_touches, level_buffer_atr


## 6. Walk-Forward Analysis

In [30]:
def walk_forward_analysis(df, train_ratio=0.7, timeframe='6M'):
    """
    Walk-forward анализ: подбор параметров на одном периоде (train),
    тестирование на следующем (test).
    """
    train_size = int(len(df) * train_ratio)
    
    train_data = df.iloc[:train_size]
    test_data = df.iloc[train_size:]
    
    print(f'Walk-Forward Analysis:')
    print(f'  Train период: {train_data.index[0]} - {train_data.index[-1]} ({len(train_data)} баров)')
    print(f'  Test период: {test_data.index[0]} - {test_data.index[-1]} ({len(test_data)} баров)')
    
    # Обучение
    bt_train = Backtest(train_data, LevelReversalStrategy,
                        cash=700, commission=0.0002)
    train_stats = bt_train.run()
    
    print(f'\n  Train результаты:')
    print(f'    Return: {train_stats["Return [%]"]:.2f}%')
    print(f'    Sharpe Ratio: {train_stats["Sharpe Ratio"]:.2f}')
    print(f'    Max Drawdown: {train_stats["Max. Drawdown [%]"]:.2f}%')
    
    # Тест с теми же параметрами
    bt_test = Backtest(test_data, LevelReversalStrategy,
                       cash=700, commission=0.0002)
    test_stats = bt_test.run()
    
    print(f'\n  Test результаты:')
    print(f'    Return: {test_stats["Return [%]"]:.2f}%')
    print(f'    Sharpe Ratio: {test_stats["Sharpe Ratio"]:.2f}')
    print(f'    Max Drawdown: {test_stats["Max. Drawdown [%]"]:.2f}%')
    
    return train_stats, test_stats

if 'df_m5' in locals() and len(df_m5) > 500:
    print('Запуск Walk-Forward Analysis...')
    train_stats, test_stats = walk_forward_analysis(df_m5)
else:
    print('⚠️  Недостаточно данных для walk-forward анализа (нужно минимум 500 баров)')

Запуск Walk-Forward Analysis...
Walk-Forward Analysis:
  Train период: 2025-04-15 21:05:00 - 2025-10-14 12:45:00 (37114 баров)
  Test период: 2025-10-14 12:50:00 - 2025-12-31 23:55:00 (15906 баров)



  Train результаты:
    Return: -16.43%
    Sharpe Ratio: -10.39
    Max Drawdown: -16.78%



  Test результаты:
    Return: -8.27%
    Sharpe Ratio: -19.17
    Max Drawdown: -8.41%


## 7. Итоговые метрики и рекомендации

In [31]:
if 'stats' in locals():
    print('\n' + '='*60)
    print('ИТОГОВЫЕ МЕТРИКИ БЭКТЕСТА')
    print('='*60)
    
    metrics = {
        'Всего прибыль (%)': stats['Return [%]'],
        'Winning trades (%)': stats['Win Rate [%]'],
        'Profit Factor': stats['Profit Factor'],
        'Sharpe Ratio': stats['Sharpe Ratio'],
        'Sortino Ratio': stats['Sortino Ratio'],
        'Max Drawdown (%)': stats['Max. Drawdown [%]'],
        'Кол-во сделок': stats['# Trades'],
        'Avg Trade (%)': stats['Avg. Trade [%]'],
        'Best Trade (%)': stats['Best Trade [%]'],
        'Worst Trade (%)': stats['Worst Trade [%]'],
    }
    
    for key, value in metrics.items():
        print(f'{key:.<40} {value:>15.2f}')
    
    print('\nРЕКОМЕНДАЦИИ:')
    if stats['Profit Factor'] > 1.5:
        print('✓ Profit Factor хороший (>1.5) - логика выглядит перспективной')
    else:
        print('⚠️  Profit Factor низкий - нужна оптимизация параметров')
    
    if stats['Max. Drawdown [%]'] < 30:
        print('✓ Просадка приемлемая (<30%) для депозита $700')
    else:
        print('⚠️  Просадка высокая (>30%) - рассмотрите более консервативные параметры SL')
    
    if stats['# Trades'] > 20:
        print(f'✓ Достаточное количество сделок ({int(stats["# Trades"])}) для статистики')
    else:
        print(f'⚠️  Сделок мало ({int(stats["# Trades"])}) - результаты могут быть нестабильны')


ИТОГОВЫЕ МЕТРИКИ БЭКТЕСТА
Всего прибыль (%).......................         -100.00
Winning trades (%)......................           35.25
Profit Factor...........................            0.52
Sharpe Ratio............................      -270408.52
Sortino Ratio...........................           -0.63
Max Drawdown (%)........................         -100.00
Кол-во сделок...........................          522.00
Avg Trade (%)...........................           -0.04
Best Trade (%)..........................            0.58
Worst Trade (%).........................           -0.59

РЕКОМЕНДАЦИИ:
⚠️  Profit Factor низкий - нужна оптимизация параметров
✓ Просадка приемлемая (<30%) для депозита $700
✓ Достаточное количество сделок (522) для статистики


## 8. График результатов

In [32]:
if 'bt' in locals():
    print('Построение графика результатов...')
    bt.plot(filename='/tmp/backtest_result.html')
    print('График сохранён в /tmp/backtest_result.html')

Построение графика результатов...
График сохранён в /tmp/backtest_result.html


## 9. Экспорт оптимальных параметров для MQL5 EA

In [33]:
# Готовые параметры для переноса в MQL5
optimal_params = {
    'ATR_PERIOD': 14,
    'MIN_TOUCHES': 3,
    'LEVEL_BUFFER_ATR': 1.0,
    'ENTRY_ZONE_ATR': 0.5,
    'FRACTAL_DEPTH': 2,
    'EMA_FAST': 50,
    'EMA_SLOW': 200,
    'ADX_THRESHOLD': 20,
    'SL_ATR_MULT': 2.0,
    'RR': 2.0,
    'LOT': 0.1,
    'USE_PIN_BAR': True,
    'USE_ENGULFING': True,
    'USE_DOJI': True,
    'ENTRY_TF': 'M5',  # или M15
}

print('\nПараметры для переноса в MQL5 EA:')
print('='*60)
for key, value in optimal_params.items():
    print(f'input {type(value).__name__:8} Inp{key:30} = {str(value):20}; // {key}')


Параметры для переноса в MQL5 EA:
input int      InpATR_PERIOD                     = 14                  ; // ATR_PERIOD
input int      InpMIN_TOUCHES                    = 3                   ; // MIN_TOUCHES
input float    InpLEVEL_BUFFER_ATR               = 1.0                 ; // LEVEL_BUFFER_ATR
input float    InpENTRY_ZONE_ATR                 = 0.5                 ; // ENTRY_ZONE_ATR
input int      InpFRACTAL_DEPTH                  = 2                   ; // FRACTAL_DEPTH
input int      InpEMA_FAST                       = 50                  ; // EMA_FAST
input int      InpEMA_SLOW                       = 200                 ; // EMA_SLOW
input int      InpADX_THRESHOLD                  = 20                  ; // ADX_THRESHOLD
input float    InpSL_ATR_MULT                    = 2.0                 ; // SL_ATR_MULT
input float    InpRR                             = 2.0                 ; // RR
input float    InpLOT                            = 0.1                 ; // LOT
input boo

In [34]:

# РУЧНАЯ ОПТИМИЗАЦИЯ ПАРАМЕТРОВ (без multiprocessing)
print('='*60)
print('РУЧНАЯ ОПТИМИЗАЦИЯ ПАРАМЕТРОВ')
print('='*60)

if 'bt' in locals() and 'df_m5' in locals():
    print('\nПоиск лучших параметров вручную...')
    print('Тестируем различные комбинации параметров:\n')
    
    # Параметры для оптимизации
    sl_mult_values = [1.0, 1.5, 2.0, 2.5, 3.0]
    rr_values = [1.5, 2.0, 2.5, 3.0]
    min_touches_values = [2, 3, 4]
    level_buffer_values = [0.5, 1.0, 1.5]
    entry_zone_values = [0.3, 0.5, 0.7]
    
    best_result = None
    best_sharpe = -float('inf')
    results = []
    
    count = 0
    total = len(sl_mult_values) * len(rr_values) * len(min_touches_values) * len(level_buffer_values) * len(entry_zone_values)
    
    for sl_mult in sl_mult_values:
        for rr in rr_values:
            if sl_mult * rr > 10:  # Constraint
                continue
            for min_touches in min_touches_values:
                for level_buffer in level_buffer_values:
                    for entry_zone in entry_zone_values:
                        count += 1
                        
                        # Запуск бэктеста с текущими параметрами
                        stats = bt.run(
                            sl_atr_mult=sl_mult,
                            rr=rr,
                            min_touches=min_touches,
                            level_buffer_atr=level_buffer,
                            entry_zone_atr=entry_zone
                        )
                        
                        sharpe = stats['Sharpe Ratio']
                        results.append({
                            'sl_atr_mult': sl_mult,
                            'rr': rr,
                            'min_touches': min_touches,
                            'level_buffer_atr': level_buffer,
                            'entry_zone_atr': entry_zone,
                            'Sharpe': sharpe,
                            'Return': stats['Return [%]'],
                            'Drawdown': stats['Max. Drawdown [%]'],
                            'Win Rate': stats['Win Rate [%]']
                        })
                        
                        if sharpe > best_sharpe:
                            best_sharpe = sharpe
                            best_result = results[-1]
                        
                        if count % 20 == 0:
                            print(f'Проверено комбинаций: {count}/{total}  |  Лучший Sharpe: {best_sharpe:.2f}')
    
    print(f'\n✅ ОПТИМИЗАЦИЯ ЗАВЕРШЕНА! Проверено {count} комбинаций\n')
    
    print('='*70)
    print('🏆 ТОП 10 ЛУЧШИХ КОМБИНАЦИЙ:')
    print('='*70)
    
    # Сортируем результаты по Sharpe Ratio
    results_df = pd.DataFrame(results).sort_values('Sharpe', ascending=False)
    
    for idx, row in results_df.head(10).iterrows():
        print(f'\n#{idx+1}:')
        print(f'  sl_atr_mult={row["sl_atr_mult"]:.1f}, rr={row["rr"]:.1f}, min_touches={int(row["min_touches"])}')
        print(f'  level_buffer={row["level_buffer_atr"]:.1f}, entry_zone={row["entry_zone_atr"]:.1f}')
        print(f'  Sharpe Ratio: {row["Sharpe"]:.2f}')
        print(f'  Return: {row["Return"]:.2f}%')
        print(f'  Max Drawdown: {row["Drawdown"]:.2f}%')
        print(f'  Win Rate: {row["Win Rate"]:.2f}%')
    
    # Сохранение лучших параметров
    optimal_params = {
        'sl_atr_mult': best_result['sl_atr_mult'],
        'rr': best_result['rr'],
        'min_touches': int(best_result['min_touches']),
        'level_buffer_atr': best_result['level_buffer_atr'],
        'entry_zone_atr': best_result['entry_zone_atr'],
    }
    
    print('\n' + '='*70)
    print('📋 ОПТИМАЛЬНЫЕ ПАРАМЕТРЫ ДЛЯ ПЕРЕНОСА В MQL5:')
    print('='*70)
    print(f'input float    InpSL_ATR_MULT               = {optimal_params["sl_atr_mult"]:.1f}  ; // SL_ATR_MULT')
    print(f'input float    InpRR                        = {optimal_params["rr"]:.1f}  ; // RR')
    print(f'input int      InpMIN_TOUCHES               = {optimal_params["min_touches"]} ; // MIN_TOUCHES')
    print(f'input float    InpLEVEL_BUFFER_ATR          = {optimal_params["level_buffer_atr"]:.1f}  ; // LEVEL_BUFFER_ATR')
    print(f'input float    InpENTRY_ZONE_ATR            = {optimal_params["entry_zone_atr"]:.1f}  ; // ENTRY_ZONE_ATR')
    print('='*70)
    
else:
    print('⚠️  Бэктест не инициализирован. Запустите ячейку 16 сначала.')


РУЧНАЯ ОПТИМИЗАЦИЯ ПАРАМЕТРОВ

Поиск лучших параметров вручную...
Тестируем различные комбинации параметров:



Проверено комбинаций: 20/540  |  Лучший Sharpe: -512599.26


Проверено комбинаций: 40/540  |  Лучший Sharpe: -497567.11


Проверено комбинаций: 60/540  |  Лучший Sharpe: -479320.50


KeyboardInterrupt: 